In [1]:
# ── Cell 0 — GPU check ────────────────────────────────────────────────
!nvidia-smi -L
import torch; assert torch.cuda.is_available(), "Set Runtime ▸ Change runtime type ▸ GPU"
print("CUDA:", torch.version.cuda, "| device:", torch.cuda.get_device_name(0))

GPU 0: NVIDIA L4 (UUID: GPU-ad42dace-5552-9718-5cfe-79a5ac72833c)
CUDA: 12.8 | device: NVIDIA L4


In [2]:
# ── Cell 1 — mount Drive + cache dir ──────────────────────────────────
from google.colab import drive; drive.mount('/content/drive')
import os
DRIVE = "/content/drive/MyDrive/haidc_m2"          # persists across sessions
os.makedirs(f"{DRIVE}/kaggle", exist_ok=True)
os.makedirs(f"{DRIVE}/artifacts", exist_ok=True)
print("cache:", DRIVE)

Mounted at /content/drive
cache: /content/drive/MyDrive/haidc_m2


In [3]:
# ── Cell 2 — clone the branch into /content/repo, then cd into it ──────
# GITHUB_TOKEN is read from Colab Secrets (🔑 left sidebar) — no re-pasting.
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
USERNAME, REPO_NAME = "Alessandro-vecchi", "Rethinking-human-AI-decision-making"
BRANCH_NAME = "claude/charming-gates-hzjyhx"
!rm -rf /content/repo
!git clone -b {BRANCH_NAME} --single-branch -q https://{GITHUB_TOKEN}@github.com/{USERNAME}/{REPO_NAME}.git /content/repo
%cd /content/repo
!git log --oneline -1

/content/repo
ac6bc4b (HEAD -> claude/charming-gates-hzjyhx, origin/claude/charming-gates-hzjyhx) M2: fix Colab runbook (consistent /content/repo, single Kaggle cell)


In [4]:
# ── Cell 3 — deps (use Colab's CUDA torch; install the rest) ──────────
# NOTE: this deviates from the pinned CPU torch 2.2.2 — expected per DECISIONS
# 2026-06-26 env note. Record the GPU torch version + a fresh lockfile hash later.
!pip -q install pandas pyarrow pyyaml scipy scikit-learn tqdm pillow
!pip -q install -e . --no-deps
import torch; print("torch", torch.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for haidc (pyproject.toml) ... done
torch 2.11.0+cu128


In [5]:
# ── Cell 4 — Kaggle creds + cached download to Drive (one upload, ever) ─
# One-time: accept the rules at kaggle.com/c/galaxy-zoo-the-galaxy-challenge/rules.
# Creds resolve from the Drive cache first; upload kaggle.json only if absent.
import os, json, pathlib, shutil
CRED = pathlib.Path(f"{DRIVE}/kaggle/kaggle.json")
if not CRED.exists():
    from google.colab import files; files.upload(); shutil.copy("kaggle.json", CRED)
creds = json.load(open(CRED))
os.environ["KAGGLE_USERNAME"] = creds["username"].strip()   # env wins over any stale file
os.environ["KAGGLE_KEY"]      = creds["key"].strip()
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(CRED, "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
!pip -q install --upgrade kaggle
C = "galaxy-zoo-the-galaxy-challenge"
for f in ["images_training_rev1.zip", "training_solutions_rev1.zip"]:
    if pathlib.Path(f"{DRIVE}/kaggle/{f}").exists():
        print("cached:", f)
    else:
        get_ipython().system(f"kaggle competitions download -c {C} -f {f} -p {DRIVE}/kaggle")
!ls -lh {DRIVE}/kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 27.2 MB/s eta 0:00:00
cached: images_training_rev1.zip
cached: training_solutions_rev1.zip
total 797M
-rw------- 1 root root 792M Dec 11  2019 images_training_rev1.zip
-rw------- 1 root root   74 Jun 27 01:31 kaggle.json
-rw------- 1 root root 4.7M Dec 11  2019 training_solutions_rev1.zip


In [6]:
# ── Cell 5 — fast local extract (Drive→local SSD, then unzip) ─────────
import glob
!cp {DRIVE}/kaggle/images_training_rev1.zip /content/imgs.zip
!unzip -q -o /content/imgs.zip -d /content/repo/data/raw/     # -> data/raw/images_training_rev1/
!cp {DRIVE}/kaggle/training_solutions_rev1.zip /content/sol.zip
!unzip -q -o /content/sol.zip -d /content/repo/data/raw/
n = len(glob.glob("/content/repo/data/raw/images_training_rev1/*.jpg"))
print("images:", n); assert n > 60000, "extract looks wrong"

images: 61578


In [7]:
# ── Cell 6 — label table (upload once to Drive, then reuse) ───────────
# label_table.parquet is git-ignored; bring it from your local data/ folder.
import pathlib
DST = "/content/repo/data/label_table.parquet"
if pathlib.Path(f"{DRIVE}/artifacts/label_table.parquet").exists():
    !cp {DRIVE}/artifacts/label_table.parquet {DST}
else:
    from google.colab import files; files.upload()             # pick label_table.parquet
    !cp label_table.parquet {DST} && cp {DST} {DRIVE}/artifacts/label_table.parquet
import pandas as pd
print(pd.read_parquet(DST).shape, "| split_manifest committed:",
      pathlib.Path("/content/repo/data/split_manifest.json").exists())

(4621, 8) | split_manifest committed: True


In [8]:
# ── Cell 7 — fidelity check: print Okati's exact image transform ──────
!git clone -q https://github.com/Networks-Learning/differentiable-learning-under-triage /content/okati || true
!grep -nE "Resize|Crop|Normalize|transforms|resize|reshape|224" /content/okati/Galaxy-zoo/prepare_data.py | head -40
# Compare against backbone.py: Resize(256)+CenterCrop(224)+ImageNet norm. If Okati differs,
# that's the first thing to fix if accuracy misses ~0.83.

7:from skimage.transform import rescale, resize, downscale_local_mean
19:X = np.zeros((num_samples,3,224,224),dtype='float')
38:        from torchvision import transforms
41:        preprocess = transforms.Compose([
42:            transforms.Resize(256),
43:            transforms.CenterCrop(224),
44:            transforms.ToTensor(),
45:            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),


In [9]:
# ── Cell 8 — run M2 (CUBLAS env makes deterministic algos legal on CUDA) ─
import os, time
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
t = time.time()
!cd /content/repo && python -m haidc.arms.backbone --config configs/backbone.yaml
print(f"elapsed {(time.time()-t)/60:.1f} min")

fatal: cannot change to 'third_party/okati2021': No such file or directory
backbone: trained on 3234 imgs, scored 694 test imgs
AI-alone test accuracy @ thr 0.5 = 0.7695 (95% CI [0.7378, 0.7983])
scores -> results/backbone_scores.parquet   artifact -> results/backbone.pt
elapsed 13.8 min


In [10]:
# ── Cell 9 — read sanity gate #2 + persist outputs ───────────────────
import json, shutil
m = json.load(open("/content/repo/results/backbone_run.json"))
print(f"AI-alone acc = {m['ai_alone_accuracy']:.4f}  CI {m['ai_alone_accuracy_ci']}")
print("≈0.83 anchor:", "PASS" if 0.80 <= m['ai_alone_accuracy'] <= 0.86 else "CHECK — see Cell 7 (Okati transform fidelity)")
# scores.parquet is the M3+ handoff (git-ignored): cache to Drive AND download to commit-adjacent.
for f in ["backbone_scores.parquet", "backbone_run.json"]:
    shutil.copy(f"/content/repo/results/{f}", f"{DRIVE}/artifacts/{f}")
from google.colab import files
files.download("/content/repo/results/backbone_scores.parquet")
files.download("/content/repo/results/backbone_run.json")

AI-alone acc = 0.7695  CI [0.7377521613832853, 0.7982708933717579]
≈0.83 anchor: CHECK — see Cell 7 (Okati transform fidelity)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>